In [1]:
from googleapiclient.discovery import build
import sys
from pathlib import Path
import os
import pandas as pd

# Load API key - TRY environment variable first, then config file
YOUTUBE_API_KEY = os.environ.get('YOUTUBE_API_KEY')

# If environment variable not set, try importing from config.py
if not YOUTUBE_API_KEY:
    try:
        from pathlib import Path
        import sys

        try:
            # Works when running as a .py file
            parent_dir = Path(__file__).resolve().parent.parent
        except NameError:
            # Fallback for Jupyter / interactive shell
            parent_dir = Path.cwd().parent

        sys.path.insert(0, str(parent_dir))
        from config import YOUTUBE_API_KEY
    except ImportError:
        pass

# If still no key, prompt user to enter it manually
if not YOUTUBE_API_KEY:
    print("=" * 50)
    print("YouTube API key not found!")
    print("Either:")
    print("  1. Set YOUTUBE_API_KEY environment variable, OR")
    print("  2. Create config.py with: YOUTUBE_API_KEY = 'your_key_here', OR")
    print("  3. Paste your API key directly below")
    print("=" * 50)
    # Uncomment the line below and paste your API key:
    # YOUTUBE_API_KEY = "paste_your_api_key_here"

# Check if we have a valid key before proceeding
if not YOUTUBE_API_KEY:
    raise ValueError("No YouTube API key provided. See instructions above.")

# Initialize YouTube client
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

print("YouTube API client initialized successfully")

YouTube API client initialized successfully


In [2]:
# Search for race highlight videos
search_query = "NASCAR Cup Series 2024 Daytona 500 highlights"

print(f"Searching YouTube for: '{search_query}'")
print("-" * 50)

# Execute search
search_request = youtube.search().list(
    q=search_query,
    part="snippet",
    maxResults=10,
    type="video",
    order="viewCount"  # Get most-viewed results
)
search_response = search_request.execute()

# Extract video IDs for detailed stats
video_ids = [item['id']['videoId'] for item in search_response['items']]
print(f"Found {len(video_ids)} videos")

Searching YouTube for: 'NASCAR Cup Series 2024 Daytona 500 highlights'
--------------------------------------------------
Found 10 videos


In [3]:
# Get detailed statistics for each video
videos_request = youtube.videos().list(
    id=','.join(video_ids),
    part="snippet,statistics"
)
videos_response = videos_request.execute()

# Process results
videos_data = []
for video in videos_response['items']:
    video_info = {
        'title': video['snippet']['title'],
        'channel': video['snippet']['channelTitle'],
        'published': video['snippet']['publishedAt'][:10],
        'views': int(video['statistics'].get('viewCount', 0)),
        'likes': int(video['statistics'].get('likeCount', 0)),
        'comments': int(video['statistics'].get('commentCount', 0)),
        'video_id': video['id']
    }
    videos_data.append(video_info)

df = pd.DataFrame(videos_data)

In [4]:
# Display results
print("\nTop videos by view count:")
print(df.sort_values('views', ascending=False)[['title', 'channel', 'views', 'likes']].to_string())

# Statistics
print(f"\nYouTube Search Statistics:")
print(f"  Total videos: {len(df)}")
print(f"  Total views: {df['views'].sum():,}")
print(f"  Average views: {df['views'].mean():,.0f}")
print(f"  Average likes: {df['likes'].mean():,.0f}")

# Check for official NASCAR channel
official = df[df['channel'].str.contains('NASCAR', case=False)]
print(f"  Official NASCAR videos: {len(official)}")


Top videos by view count:
                                                                                               title                   channel     views  likes
0                                                             Dale Earnhardt's fatal crash @ Daytona              NascarAllOut  10896917  62253
1                               NASCAR All-Star Race from Bristol Motor Speedway | NASCAR Cup Series                    NASCAR   4438260  29125
7        Lets watch the final lap of the Daytona 500 from the pov of Tyler Reddick 🏁 #nascar #racing       Nascar Hall Of Fame   3200169  42877
3                                                   The NASCAR Rookie Who Was KICKED Out Mid Season!  Ultimate Diecast Review!   3059459  51054
2                                                        2024 Daytona 500 Highlights | NASCAR on FOX             NASCAR on FOX   2450639  28314
6                                            SCARY CRASH AT DAYTONA  #nascar #racing #crash #daytona         

In [5]:
import os

# Create directory if it doesn't exist
os.makedirs('data/raw', exist_ok=True)

# Save to CSV
df.to_csv('data/raw/test_youtube_videos.csv', index=False)
print(f"\nTest data saved to data/raw/test_youtube_videos.csv")

# Note quota usage
print("\nQuota used: ~200 units (search: 100 + videos: 100)")
print("Remaining daily quota: ~9,800 units")


Test data saved to data/raw/test_youtube_videos.csv

Quota used: ~200 units (search: 100 + videos: 100)
Remaining daily quota: ~9,800 units
